In [1]:
import re

def preprocess_text(text):


    text = text.strip()


    text = re.sub(r'<[^>]+>', '', text)


    text = re.sub(r'http\S+|www\S+', '', text)


    text = re.sub(r'\s+', ' ', text)


    text = re.sub(r'[^a-zA-Z0-9\s.,!?\'\"%-]', '', text)


    if len(text) > 512:
        text = text[:512]


    text = text.strip()

    return text

In [2]:
sample_text = """
   <p>Severe <b>floods</b> in India!!   have destroyed
   wheat crops across Punjab.\n\nVisit https://agrinews.com
   for more@@ details###.
   Farmers reported up to 40% yield loss this season.   </p>
"""

cleaned = preprocess_text(sample_text)

print("ORIGINAL TEXT:")
print(repr(sample_text))
print()
print("CLEANED TEXT:")
print(cleaned)
print()
print("Original length:", len(sample_text))
print("Cleaned length :", len(cleaned))

ORIGINAL TEXT:
'\n   <p>Severe <b>floods</b> in India!!   have destroyed   \n   wheat crops across Punjab.\n\nVisit https://agrinews.com \n   for more@@ details###.\n   Farmers reported up to 40% yield loss this season.   </p>\n'

CLEANED TEXT:
Severe floods in India!! have destroyed wheat crops across Punjab. Visit for more details. Farmers reported up to 40% yield loss this season.

Original length: 205
Cleaned length : 141


# **Step 3 — spaCy NER Function**

In [3]:
import spacy

nlp = spacy.load("en_core_web_sm")

commodity_fallback = {
    "paddy": "rice",
    "groundnut": "peanut oil",
    "guar": "guar gum",
    "maize": "corn",
    "crude": "crude oil",
    "soybean": "soybean",
    "palm": "palm oil",
    "sugar": "sugar",
    "cotton": "cotton",
    "coffee": "coffee"
}

def extract_entities(text):
    doc = nlp(text)

    countries = []
    commodities = []
    organizations = []
    dates = []

    for ent in doc.ents:
        if ent.label_ == "GPE":
            countries.append({"name": ent.text, "start": ent.start_char, "end": ent.end_char})
        elif ent.label_ == "PRODUCT":
            commodities.append({"name": ent.text, "start": ent.start_char, "end": ent.end_char})
        elif ent.label_ == "ORG":
            organizations.append({"name": ent.text, "start": ent.start_char, "end": ent.end_char})
        elif ent.label_ == "DATE":
            dates.append({"name": ent.text, "start": ent.start_char, "end": ent.end_char})

    for token in doc:
        word = token.text.lower()
        if word in commodity_fallback:
            already_found = any(c["name"].lower() == word for c in commodities)
            if not already_found:
                commodities.append({"name": commodity_fallback[word], "start": token.idx, "end": token.idx + len(token.text)})

    return {
        "countries": countries,
        "commodities": commodities,
        "organizations": organizations,
        "dates": dates
    }

In [4]:
cleaned_text = preprocess_text(sample_text)

entities = extract_entities(cleaned_text)

print("CLEANED TEXT USED:")
print(cleaned_text)
print()
print("COUNTRIES FOUND:")
for c in entities["countries"]:
    print(" -", c["name"], "| position:", c["start"], "to", c["end"])

print()
print("COMMODITIES FOUND:")
for c in entities["commodities"]:
    print(" -", c["name"], "| position:", c["start"], "to", c["end"])

print()
print("ORGANIZATIONS FOUND:")
for o in entities["organizations"]:
    print(" -", o["name"])

print()
print("DATES FOUND:")
for d in entities["dates"]:
    print(" -", d["name"])

CLEANED TEXT USED:
Severe floods in India!! have destroyed wheat crops across Punjab. Visit for more details. Farmers reported up to 40% yield loss this season.

COUNTRIES FOUND:
 - India | position: 17 to 22

COMMODITIES FOUND:

ORGANIZATIONS FOUND:

DATES FOUND:
 - this season


# **Step 4 — Zero-Shot Risk Classification Function**

In [5]:
from transformers import pipeline

risk_classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

risk_labels = [
    "weather disaster",
    "political sanction",
    "logistics disruption",
    "price surge",
    "supply shortage",
    "no risk"
]

def classify_risk(text):
    result = risk_classifier(text, candidate_labels=risk_labels)

    top_label = result["labels"][0]
    top_score = round(result["scores"][0], 4)

    if top_score > 0.75:
        severity = "HIGH"
    elif top_score > 0.50:
        severity = "MEDIUM"
    else:
        severity = "LOW"

    all_scores = {}
    for label, score in zip(result["labels"], result["scores"]):
        all_scores[label] = round(score, 4)

    return {
        "risk_type": top_label,
        "risk_score": top_score,
        "severity": severity,
        "all_scores": all_scores
    }

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

# **Step 5 — Sentiment Analysis Function**

In [6]:
sentiment_analyzer = pipeline("sentiment-analysis")

def analyze_sentiment(text):
    result = sentiment_analyzer(text[:512])

    label = result[0]["label"]
    score = round(result[0]["score"], 4)

    return {
        "sentiment": label,
        "sentiment_score": score
    }

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [7]:
cleaned_text = preprocess_text(sample_text)

sentiment_result = analyze_sentiment(cleaned_text)

print("TEXT ANALYZED:")
print(cleaned_text)
print()
print("SENTIMENT       :", sentiment_result["sentiment"])
print("SENTIMENT SCORE :", sentiment_result["sentiment_score"])

TEXT ANALYZED:
Severe floods in India!! have destroyed wheat crops across Punjab. Visit for more details. Farmers reported up to 40% yield loss this season.

SENTIMENT       : NEGATIVE
SENTIMENT SCORE : 0.9987


# **Step 6 — Final Merge Function**

In [8]:
def analyze_news(raw_text):

    cleaned_text = preprocess_text(raw_text)

    if len(cleaned_text) < 20:
        return {"error": "Text is too short. Please provide at least 20 characters."}

    entities = extract_entities(cleaned_text)
    risk_result = classify_risk(cleaned_text)
    sentiment_result = analyze_sentiment(cleaned_text)

    country = entities["countries"][0]["name"] if entities["countries"] else "Not found"
    commodity = entities["commodities"][0]["name"] if entities["commodities"] else "Not found"

    if country != "Not found" and commodity != "Not found":
        summary = f"{risk_result['severity']} {risk_result['risk_type']} detected for {commodity} in {country}"
    elif commodity != "Not found":
        summary = f"{risk_result['severity']} {risk_result['risk_type']} detected for {commodity}"
    elif country != "Not found":
        summary = f"{risk_result['severity']} {risk_result['risk_type']} detected in {country}"
    else:
        summary = f"{risk_result['severity']} {risk_result['risk_type']} detected"

    return {
        "cleaned_text": cleaned_text,
        "entities": entities,
        "risk_type": risk_result["risk_type"],
        "risk_score": risk_result["risk_score"],
        "severity": risk_result["severity"],
        "all_scores": risk_result["all_scores"],
        "sentiment": sentiment_result["sentiment"],
        "sentiment_score": sentiment_result["sentiment_score"],
        "summary": summary
    }

In [9]:
import json

news_samples = [
    "Severe floods in Pakistan have destroyed standing rice crops across Sindh and Punjab provinces, with initial estimates suggesting a loss of over 2 million tonnes of output.",

    "The European Union has imposed fresh sanctions on Russian fertilizer exports, raising concerns about global supply disruptions for wheat-producing nations dependent on Russian inputs.",

    "A prolonged strike at Chiles major copper mining operations has halted shipments for over two weeks, leaving buyers in Asia scrambling for alternative sources.",

    "India recorded its best wheat harvest in five years this Rabi season, with the Food Corporation of India reporting procurement levels well above government targets.",

    "Crude oil prices jumped nearly 8% this week after OPEC announced an unexpected production cut of 1.5 million barrels per day, catching markets off guard."
]

for i, news in enumerate(news_samples, 1):
    print(f"{'='*60}")
    print(f"SAMPLE {i}")
    print(f"{'='*60}")
    result = analyze_news(news)
    print("SUMMARY    :", result["summary"])
    print("COUNTRY    :", result["entities"]["countries"][0]["name"] if result["entities"]["countries"] else "Not found")
    print("COMMODITY  :", result["entities"]["commodities"][0]["name"] if result["entities"]["commodities"] else "Not found")
    print("RISK TYPE  :", result["risk_type"])
    print("RISK SCORE :", result["risk_score"])
    print("SEVERITY   :", result["severity"])
    print("SENTIMENT  :", result["sentiment"])
    print("SENT SCORE :", result["sentiment_score"])
    print()

SAMPLE 1
SUMMARY    : HIGH weather disaster detected in Pakistan
COUNTRY    : Pakistan
COMMODITY  : Not found
RISK TYPE  : weather disaster
RISK SCORE : 0.8778
SEVERITY   : HIGH
SENTIMENT  : NEGATIVE
SENT SCORE : 0.9995

SAMPLE 2
SUMMARY    : LOW supply shortage detected
COUNTRY    : Not found
COMMODITY  : Not found
RISK TYPE  : supply shortage
RISK SCORE : 0.4934
SEVERITY   : LOW
SENTIMENT  : NEGATIVE
SENT SCORE : 0.9896

SAMPLE 3
SUMMARY    : HIGH supply shortage detected
COUNTRY    : Not found
COMMODITY  : Not found
RISK TYPE  : supply shortage
RISK SCORE : 0.8339
SEVERITY   : HIGH
SENTIMENT  : NEGATIVE
SENT SCORE : 0.9994

SAMPLE 4
SUMMARY    : LOW price surge detected in India
COUNTRY    : India
COMMODITY  : Not found
RISK TYPE  : price surge
RISK SCORE : 0.4502
SEVERITY   : LOW
SENTIMENT  : POSITIVE
SENT SCORE : 0.9951

SAMPLE 5
SUMMARY    : HIGH price surge detected for crude oil
COUNTRY    : Not found
COMMODITY  : crude oil
RISK TYPE  : price surge
RISK SCORE : 0.9401
SEVERITY 